## 2장 1강 : System·User·Assistant 메시지 구조

### 3. 역할별 메시지 템플릿 조립하기

- system : 모델의 역할 (페르소나), 상황, 과업, 규칙 정의, 최우선순위 / SystemMessage
- user : 실제 질의 응답에 대한 사용자 질문, 처리할 데이터의 전달, 가변적, 프롬프트 템플 / HumanMessage
- assistant : AI가 출력한 응답 / AIMessage

- user - assistant -> 대화의 맥락 정보를 쌍으로 전달, 한 쌍 - 턴, 여러 쌍 - 멀티 턴
- BaseMessage의 하위 클래스 (HumanMessage, SystemMessage, AIMessage, ToolMessage)

#### 3.1 라이브러리 로드 및 환경 설정

In [15]:
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage


model = init_chat_model(
    model_provider = "Ollama",
    model="mistral"
)

parser = StrOutputParser()

chain = model | parser


#### 3.2 역할별 메시지 템플릿 생성 및 실행

In [11]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 재치있게 컴퓨터의 개념을 설명하는 유치원 교사입니다."),
    ("user", "다음 개념을 설명하세요: {text}")
])

chain2 = prompt_template | model | parser

response = chain2.invoke({"text": "함수"})


response

' 컴퓨터 프로그램을 만들고 있는데, 당신은 동일한 일을 반복하고 있어요! 이제 당신은 이런 일을 더 간편하고 효율적으로 하기 위해서 함수라는 개념을 알아보아요!\n\n함수는 컴퓨터 프로그램에서 특정 작업을 수행하는 독자적인 덩어리입니다. 함수를 만들면, 이 함수를 사용하면 같은 작업을 반복할 필요가 없어져요!\n\n예를 들어, 컴퓨터 프로그램에서 두 수의 합을 계산해야 한다고 해봅시다. 당신이 두 수를 입력하고 두 수를 더하고 그 결과를 출력하는 코드를 여러 번 만드시면 어떨까요? 이렇게 하면 코드가 많아지고 오류가 발생할 가능성도 높아집니다.\n\n하지만, 함수를 사용하면 이런 문제를 해결할 수 있습니다! 함수를 만들고 두 수를 입력받고 두 수를 더하고 결과를 출력하는 코드를 한 번 만들고 나면, 함수를 호출하면 이 코드가 자동으로 실행되니까요!\n\n이렇게 함수를 사용하면 코드를 재사용하고 유지 관리하기가 쉬워지고 프로그램의 효율성도 높아집니다.\n\n이제 당신이 만든 함수를 이용해서 두 수의 합을 계산해봅시다! 함수를 만드는 방법에 대해서는 따로 알아보시면 됩니다.'

### 4. 대화 이력(Memory) 구조 설계하기

#### 4.1 Assistant 메시지 기반 문맥 유지

In [18]:
messages : list[BaseMessage] = [
    HumanMessage("내 이름 김철수야, 만나서 반가워"),
    AIMessage ("반갑습니다. 철수님, 무엇을 도와드릴까요?"),
    HumanMessage("오늘 점심 메뉴를 추천해줘"),
    AIMessage("오늘 점심은 돈까스가 좋을 것 같아요."),
    HumanMessage("오늘 점심 추천이 뭐였지?"),
    #HumanMessage("내 이름이 뭐라고 했지?")
]
  
response = chain.invoke(messages)

response

' 오늘 점심 추천이 돈까스였습니다.'